### 超参数的存储

超参数可以选择存储在 yaml 配置文件中，方便统一管理

需要使用时通过加载配置文件，获取参数值

若配置文件超参数如下：

```yaml
# config.yaml
model_size: 256
hidden_size: 1024
```

In [ ]:
import yaml

# 加载配置文件
with open('config.yaml', 'r') as config_file:
    params = yaml.safe_load(config_file)

# 获取对应超参数
MODEL_SIZE = params['model_size']
HIDDEN_SIZE = params['hidden_size']

<br>

---

### 多GPU训练（DDP）


<br>

---

### 常用的数据保存技巧

- nn.Module 模型保存

In [ ]:
import torch

# 模拟需要保存的模型
model = torch.nn.Linear(256, 1024)

# 将模型参数保存为 pth 文件
# 保存内容包括参数名、参数值
# 并不保存模型结构
torch.save(model.state_dict(), 'model.pth')

# 加载模型，都先默认加载到 CPU 上
# 后续的推理需要在哪里进行，使用 to device 函数即可
state_dict = torch.load('model.pth', map_location='cpu')
model.load_state_dict(state_dict)

In [ ]:
# 若模型使用多 GPU 训练
main_device = torch.device('cuda:0')
device_ids = [0, 1, 2, 3]
model = model.to(main_device)
model = torch.nn.DataParallel(model, device_ids=device_ids)

# 此时 model 是外壳，真正需要保存的是 model.module
torch.save(model.module.state_dict(), "model.pth")

# 加载操作同上
state_dict = torch.load('model.pth', map_location='cpu')
model.load_state_dict(state_dict)

- numpy 数组

In [ ]:
import numpy as np

arr = np.array([1, 2, 3], [4, 5, 6])

# 保存为本地的 npy 文件
np.save('arr.npy', arr)

# 加载为 numpy 数组
arr_ = np.load('arr.npy')

<br>

---

### BPE模型训练

BPE 用于文本序列的 tokenization，主要训练流程如下：

![](./md-img/bpe.png)

有实用的 API 调用，如下：

In [ ]:
# 语料
lines = [
    'Hello, world!',
    'Good afternoon.',
    'I love you',
    '...'
]

# 首先汇总语料库，将整个训练集的语料集中到一个 txt 文件中
with open('corpus.txt', 'w', encoding='utf-8') as corpus:
    for line in lines:
        line = line.strip()
        if line:
            corpus.write(line + '\n')

# 使用语料库训练 BPE 模型
import sentencepiece

# 保存 BPE 模型的文件名，不带后缀
# 后续会依据这个名，保存两个文件，后缀分别为 '.model' 和 '.vocab'
bpe_prefix = '/BPE_MODEL/english_bpe'   
sentencepiece.SentencePieceTrainer.train(
    input='corpus.txt',        # 指定训练的语料库
    model_prefix=bpe_prefix,   # 指定模型保存的名字
    vocab_size=100,            # 指定BPE训练的词表大小
    pad_id=3,                  # 指定pad的id，不指定默认没有pad
)

# 加载训练好的 BPE 模型
sp = sentencepiece.SentencePieceProcessor(model_file=f'{bpe_prefix}.model')
vocab_size = sp.get_piece_size()

# 查看词表键值对
for i in range(vocab_size):
    print(f'{i}: {sp.id_to_piece(i)}')

# 对文本进行编码和解码
ids = sp.encode('Hello, world!')
text = sp.decode(ids)

<br>

---

### BLEU指标计算

在使用测试集的 BLEU 分数来评估模型效果时，需要要使用整个测试集的机器翻译来进行计算

测试集每一个样本对应一个机器翻译，每一个样本可以对应多个参考翻译

In [ ]:
from sacrebleu import corpus_bleu

# 3 条机器翻译结果
predicts = [
    "I like apples",
    "The cat sits on the mat",
    "This is a test"
]

# 参考翻译
references = [
    ["I love apples", "I enjoy apples"],                         # 第一条机器翻译对应的所有参考翻译
    ["The cat is sitting on the mat", "A cat sits on the mat"],  # 第二条机器翻译对应的所有参考翻译
    ["This is just a test", "This is a simple test"]             # 第三条机器翻译对应的所有参考翻译
]

# 最终计算的 BLEU 分数，'13a'分词方式适用于英语
bleu = corpus_bleu(predicts, references, tokenize='13a')
bleu_score = bleu.score

分词方式介绍：

- 13a，英语分词

- zh，中文分词

- intl，国际语言分词

- ja-mecab，日语分词

- zh_CN，中文分词（第二个方案）

使用不同的分词方案，得到的 BLEU 分数是不同的

<br>

---

### 标签平滑化

标签平滑化就是将原本标签的one-hot形式的概率分布进行变换

让标签的表达不要过于绝对，可以有效防止模型表达过于自信

In [ ]:
batch_size = 128
seq_len = 30
vocab_size = 100

# 模拟标签数据
label = torch.randint(0, vocab_size, (batch_size, seq_len))
one_hot_label = torch.nn.functional.one_hot(label, vocab_size)


smooth_rate = 0.1                              # 平滑率
confidence = 1 - smooth_rate                   # 真类平滑后的概率
smooth_value = smooth_rate / (vocab_size - 1)  # 其他类平滑后的概率

# 平滑标签
smoothed_label = one_hot_label * (confidence - smooth_value) + smooth_value

<br>

---

### Mask 的应用

- 注意力机制中 Mask 的应用

在计算某一个查询对应的结果时，可能不想考虑某些键值对

这时只需要将这个查询计算出来的这几个键值对的注意力分数置为 -inf

这样在通过 softmax 层得到的对应这几个键值对的注意力权重就是 0 了

In [ ]:
from torch import nn, Tensor
import math

class MultiHeadAttention(nn.Module):
    def __init__(self, query_size: int, key_size: int, value_size: int, hidden_size: int, num_heads: int):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads
        self.linear_q = nn.Linear(query_size, hidden_size)
        self.linear_k = nn.Linear(key_size, hidden_size)
        self.linear_v = nn.Linear(value_size, hidden_size)
        self.linear_o = nn.Linear(hidden_size, hidden_size)
        self.softmax = nn.Softmax(dim=-1)

    # query: (batch_size, num_querys, query_size)
    # key: (batch_size, num_pairs, key_size)
    # value: (batch_size, num_pairs, value_size)
    # mask: (batch_size, num_querys, num_pairs)
    def forward(self, query: Tensor, key: Tensor, value: Tensor, mask: Tensor = None) -> Tensor:
        batch_size = query.shape[0]
        num_querys = query.shape[1]
        num_pairs = key.shape[1]

        query = self.linear_q(query)
        key = self.linear_k(key)
        value = self.linear_v(value)

        query = query.reshape(batch_size, num_querys, self.num_heads, self.head_dim).permute(0, 2, 1, 3).reshape(batch_size * self.num_heads, num_querys, self.head_dim)
        key = key.reshape(batch_size, num_pairs, self.num_heads, self.head_dim).permute(0, 2, 3, 1).reshape(batch_size * self.num_heads, self.head_dim, num_pairs)
        value = value.reshape(batch_size, num_pairs, self.num_heads, self.head_dim).permute(0, 2, 1, 3).reshape(batch_size * self.num_heads, num_pairs, self.head_dim)
        score = torch.bmm(query, key) / math.sqrt(self.head_dim)

        # 依据 Mask，保留 True 对应位置的数据
        # 将 False 对应位置的数据置为 -inf
        if mask is not None:
            mask = mask.unsqueeze(1).expand(batch_size, self.num_heads, num_querys, num_pairs).reshape(batch_size * self.num_heads, num_querys, num_pairs)
            score = score.masked_fill(mask == False, float('-inf'))

        weight = self.softmax(score)
        output = torch.bmm(weight, value)
        output = output.reshape(batch_size, self.num_heads, num_querys, self.head_dim).permute(0, 2, 1, 3).reshape(batch_size, num_querys, self.hidden_size)
        return self.linear_o(output)

- 损失函数中 Mask 的应用

在计算损失函数时，有时我们需要丢弃一部分无关的损失，防止模型走偏

In [ ]:
class MaskedCrossEntropyLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.softmax = nn.Softmax(dim=-1)

    # pred: (batch_size, seq_len, vocab_size)
    # label: (batch_size, seq_len, vocab_size)
    # mask: (batch_size, seq_len)
    def forward(self, pred: Tensor, label: Tensor, mask: Tensor = None) -> Tensor:
        log_prob = torch.log_softmax(pred, dim=-1)
        loss = -(label * log_prob).sum(dim=-1)

        # 保留 True 对应位置的损失
        # 丢弃 False 对应位置的损失
        if mask is not None:
            loss = loss * mask

        # 在计算平均值时，总数也只能是考虑的损失的数量
        return loss.sum() / mask.sum()

<br>

---

### warmup学习率调度

将选用的优化器传入

在需要进行优化时，先使用学习率调度器改变优化器内置的学习率

然后再调用优化器进行参数优化

In [ ]:
class LRScheduler:
    def __init__(self, optimizer, warmup_steps, model_size):
        self.optimizer = optimizer
        self.warmup_steps = warmup_steps
        self.lr_mul = model_size ** (-0.5)
        self.step_num = 0

    def step(self):
        self.step_num += 1
        lr = self.lr_mul * min(self.step_num ** (-0.5), self.step_num * (self.warmup_steps ** -1.5))
        for param_group in self.optimizer.param_groups:
            param_group["lr"] = lr
        return lr

<br>

---

### 混合精度训练

在训练过程中的正向传播、损失计算使用 torch.float16 计算

可以节省显存、加速训练

在反向传播计算梯度时通过放大损失，防止 torch.float16 类型下溢，导致梯度消失

然后还原梯度倍率后进行参数更新

In [ ]:
from torch.amp import autocast, GradScaler
from torch.optim import Adam

model = nn.Linear(256, 1024)
scaler = GradScaler()
optimizer = Adam(
    params=model.parameters(),
    lr=0.001,
    betas=(0.9, 0.98),
    eps=1e-9
)

In [ ]:
model.train()
for epoch in range(1, 30 + 1):
    for step in range(1, 100 + 1):
        sample = torch.randn(10, 256)
        optimizer.zero_grad()
        with autocast(device_type='cpu', dtype=torch.float16):
            output = model.forward(sample)
            loss = output.sum()    # 这个损失函数乱写的

        scaler.scale(loss).backward()   # 根据缩放倍率放大 loss 计算梯度
        scaler.step(optimizer)          # 还原梯度倍率进行安全的参数优化
        scaler.update()                 # 根据本次参数更新状况调整缩放倍率